# 🔎 Project: A Multi-User Research Agent with SQL Memory

## Learning Objectives
In this notebook, you will learn:
1. **Custom tools** - a web-search-and-extract tool and a weather tool
2. **A tool-calling agent** - a system prompt with history and scratchpad placeholders, run by `AgentExecutor`
3. **Multi-user SQL memory** - wrap the agent with `RunnableWithMessageHistory` and `SQLChatMessageHistory`
4. **Contextual follow-ups** - answer "What about Intel?" and "Which city is hotter?" from each user's history

## Where this fits
The last application of the module: the per-user SQLite memory from
`04_multi_user_sessions_and_persistence.ipynb`, applied to an agent that calls tools. Building agents with
tools is taught in repository 06 (agent fundamentals); here the focus is the memory.

## Prerequisites
- `04_multi_user_sessions_and_persistence.ipynb`; tool-calling agents (repository 06)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY`, `TAVILY_API_KEY` and `WEATHER_API_KEY` (OpenWeatherMap) in the project-root `.env`

> **⚠️ Legacy API**: `AgentExecutor` and `create_tool_calling_agent` are LangChain 0.x APIs kept in `langchain_classic`; `RunnableWithMessageHistory` is deprecated in `langchain-core` 1.3.3 and removed in 2.0; `SQLChatMessageHistory` comes from the sunset `langchain-community`. LangChain now recommends LangGraph persistence instead — a checkpointer
> and a `thread_id` — taught in [`01_memory_and_conversational_agent.ipynb`](../notebooks/06a_memory_and_conversational_agent.ipynb)
> (module 02). This module keeps the older API because existing LCEL code uses it everywhere; the setup cell
> lets its deprecation warnings through so you see them where the deprecated classes are used.


## 📚 Conceptual Overview

```
┌──────────────────────────────────────────────────────────────┐
│               Multi-User Conversational Agent                │
├──────────────────────────────────────────────────────────────┤
│  User 1: john001            │  User 2: bond007               │
│  "Nvidia earnings?"         │  "Weather in Bangalore?"       │
│  "What about Intel?"        │  "What about Mumbai?"          │
│  "Which is doing better?"   │  "Which city is hotter?"       │
│              │              │              │                 │
│              ▼              │              ▼                 │
│  ┌────────────────────────────────────────────────────────┐  │
│  │                SQLite memory database                  │  │
│  │  john001 → [message1, message2, …]                     │  │
│  │  bond007 → [message1, message2, …]                     │  │
│  └────────────────────────────────────────────────────────┘  │
└──────────────────────────────────────────────────────────────┘
```

![](https://i.imgur.com/lHWqaT9.png)

---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library warnings — except LangChain's deprecation warnings, which it lets
through because this notebook uses deprecated APIs on purpose.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv
from langchain_core._api import LangChainDeprecationWarning

warnings.filterwarnings("ignore")
# This notebook uses deprecated LangChain APIs on purpose: let their deprecation warnings through
warnings.filterwarnings("default", category=LangChainDeprecationWarning)

# Reads OPENAI_API_KEY, TAVILY_API_KEY and WEATHER_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the agent's reasoning and tool selection
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🛠️ Part 1: Custom Tools

1. **`search_web_extract_info`**: searches with Tavily, then downloads each result page and converts it to text
2. **`get_weather`**: current weather for a city from the OpenWeatherMap API

### Key Concepts:
- The `@tool` decorator turns a function into a LangChain tool; its docstring tells the model when to use it
- Every network call has a timeout, and slow page downloads are abandoned rather than awaited

![](https://i.imgur.com/TyPAYXE.png)

In [ ]:
# ============================================================================
# PART 1: WEB SEARCH + EXTRACTION AND WEATHER TOOLS
# ============================================================================
import os
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import TimeoutError as FutureTimeout

import requests
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from markitdown import MarkItDown

tavily_tool = TavilySearch(max_results=5, search_depth="advanced", include_answer=False)

session = requests.Session()
session.headers.update({"User-Agent": "aied-08-research-agent-notebook"})
md_converter = MarkItDown(requests_session=session)


@tool
def search_web_extract_info(query: str) -> list:
    """Search the web for a query and extract useful information from the result pages."""
    print(f"  [tool] web search: {query}")
    results = tavily_tool.invoke({"query": query}).get("results", [])

    def extract_content(url: str) -> str:
        page = md_converter.convert(url)
        return f"{(page.title or '').strip()}\n{page.text_content.strip()}"

    docs = []
    executor = ThreadPoolExecutor(max_workers=5)
    futures = {executor.submit(extract_content, r["url"]): r for r in results}
    for future, result in futures.items():
        try:
            docs.append(future.result(timeout=30))
        except FutureTimeout:
            docs.append(result.get("content", ""))  # fall back to Tavily's snippet
        except Exception as error:
            print(f"  [tool] could not extract {result['url']}: {error}")
            docs.append(result.get("content", ""))
    executor.shutdown(wait=False, cancel_futures=True)  # never wait for a hung download
    return docs


@tool
def get_weather(query: str) -> dict | str:
    """Fetch current weather data (temperature, humidity, wind) for an Indian city from OpenWeatherMap."""
    print(f"  [tool] weather: {query}")
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {"q": f"{query},IN", "appid": os.getenv("WEATHER_API_KEY"), "units": "metric"}
    data = requests.get(url, params=params, timeout=10).json()
    return data if data.get("name") else "Weather Data Not Found"


tools = [search_web_extract_info, get_weather]
print("✅ Tools:", [t.name for t in tools])

---
## 🤖 Part 2: The Agent

The prompt has two placeholders:
- **`history`** (optional): where `RunnableWithMessageHistory` injects the user's past conversation
- **`agent_scratchpad`**: the agent's own tool calls and results within one question

> **Note**: `create_tool_calling_agent` and `AgentExecutor` are LangChain 0.x APIs, kept in `langchain_classic`.
> LangChain 1.x's `create_agent` runs its own loop and takes memory through a checkpointer instead of
> `RunnableWithMessageHistory`. Module 02 teaches that persistence model on a hand-built LangGraph agent
> ([`01_memory_and_conversational_agent.ipynb`](../notebooks/06a_memory_and_conversational_agent.ipynb)).

In [ ]:
# ============================================================================
# PART 2: A TOOL-CALLING PROMPT AND THE AGENT EXECUTOR
# ============================================================================
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYS_PROMPT = """Act as a helpful assistant.
Call a tool when it helps, and answer from your own knowledge when you can.
Break a task into smaller steps if needed.

Tools at your disposal:
  - get_weather: whenever the user asks for the weather of a place.
  - search_web_extract_info: whenever the user asks for specific or recent information you don't know.
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYS_PROMPT),
    MessagesPlaceholder(variable_name="history", optional=True),  # per-user memory goes here
    ("human", "{query}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),        # the agent's tool loop
])

agent = create_tool_calling_agent(llm, tools, prompt_template)
agent_executor = AgentExecutor(agent=agent, tools=tools, max_iterations=10)
print("✅ Agent and AgentExecutor created")

---
## 🧠 Part 3: Multi-User SQL Memory

The agent executor is a runnable, so the same wrapper as in `04_multi_user_sessions_and_persistence.ipynb`
gives it one SQLite-backed history per user. `RESET_MEMORY = True` makes the demo repeatable.

In [ ]:
# ============================================================================
# PART 3: WRAP THE AGENT WITH PER-USER HISTORY
# ============================================================================
from IPython.display import Markdown, display
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

DB_FILE = "research_agent_memory.db"
RESET_MEMORY = True
if RESET_MEMORY and os.path.exists(DB_FILE):
    os.remove(DB_FILE)


def get_session_history_db(session_id: str) -> SQLChatMessageHistory:
    # pass `connection` by keyword: the 2nd positional argument is table_name
    return SQLChatMessageHistory(session_id=session_id, connection=f"sqlite:///{DB_FILE}")


agentic_chatbot = RunnableWithMessageHistory(
    agent_executor,
    get_session_history_db,
    input_messages_key="query",
    history_messages_key="history",
    output_messages_key="output",  # AgentExecutor returns a dict; save its 'output' as the AI message
)


def chat_with_agent(prompt_text: str, session_id: str) -> None:
    print(f"{session_id}: {prompt_text}")
    response = agentic_chatbot.invoke({"query": prompt_text}, {"configurable": {"session_id": session_id}})
    display(Markdown(response["output"]))


print("✅ Multi-user conversational agent ready")

---
## 🧪 Part 4: Contextual Follow-ups for Two Users

### User `john001` — financial research
"What about Intel?" only makes sense with the earlier question about earnings calls.

In [ ]:
# ============================================================================
# PART 4A: john001 — THREE TURNS THAT BUILD ON EACH OTHER
# ============================================================================
chat_with_agent("Summarize the key points discussed in Nvidia's Q4 2024 earnings call", "john001")
chat_with_agent("What about Intel?", "john001")
chat_with_agent("Which company seems to be doing better?", "john001")

The second search query printed by the tool should be about Intel's *Q4 2024 earnings*, not Intel in general,
and the third answer should compare the two companies without searching again — both only possible with history.

### User `bond007` — weather
A different user with a separate history.

In [ ]:
# ============================================================================
# PART 4B: bond007 — A SEPARATE CONVERSATION
# ============================================================================
chat_with_agent("How is the weather in Bangalore today? Show detailed statistics", "bond007")
chat_with_agent("What about Mumbai?", "bond007")
chat_with_agent("Which city is hotter?", "bond007")

| Question | Without memory | With memory |
|---|---|---|
| "What about Intel?" | ❌ asks what you mean | ✅ searches for Intel's Q4 2024 earnings |
| "Which company is doing better?" | ❌ "which companies?" | ✅ compares Nvidia and Intel |
| "Which city is hotter?" | ❌ "which cities?" | ✅ compares Bangalore and Mumbai |

---
## 📝 Summary

In this notebook, we learned:

### 1. Custom tools
- Tavily search plus page extraction, and a weather API tool, each with timeouts

### 2. A tool-calling agent
- A prompt with `history` and `agent_scratchpad` placeholders, executed by `AgentExecutor`

### 3. Multi-user SQL memory
- `RunnableWithMessageHistory` + `SQLChatMessageHistory` give the agent one persistent history per user

### 4. Contextual follow-ups
- Short follow-ups are resolved from each user's own history

### Next Steps
- **Module 02 — `02_langgraph_threads_and_checkpoints`**: the modern replacement — LangGraph threads and checkpointers